In [3]:
### Importy i wczytanie danych
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split

# --- Ustalanie ścieżek względem notebooka ---
NOTEBOOK_DIR = Path(__file__).resolve().parent if "__file__" in locals() else Path.cwd()
DATA_CSV = NOTEBOOK_DIR.parent / "data" / "plants_clean_final.csv"
FEATURE_LIST = NOTEBOOK_DIR.parent / "artifacts" / "feature_list.json"

# --- Wczytanie danych ---
df = pd.read_csv(DATA_CSV, encoding="cp1250")
df.loc[:2, ["tempmin_pasmo", "tempmax_pasmo", "category_group", "light_level_clean"]]


print("✅ Wczytano dane z:", DATA_CSV)
print("Shape:", df.shape)

# --- Szybki podgląd ---
display(df.head(3))
df.info()


✅ Wczytano dane z: c:\Users\annas\projekty_infoshare\projekt ML\data\plants_clean_final.csv
Shape: (159, 17)


,latin,category,ideallight,toleratedlight,watering,tempmax.celsius,tempmin.celsius,toxicity_animals,toxicity_humans,tempmin_pasmo,tempmax_pasmo,is_cold_tolerant,is_heat_tolerant,category_group,light_level_clean,watering_group,toxicity_any
0,Aeschynanthus lobianus,Hanging,Bright light,Direct sunlight,Keep moist between watering. Can be a bit dry ...,32,14,nietoksyczna,nietoksyczna,Lubi ciepĹ‚o,Normalna tolerancja ciepĹ‚a,0,0,WiszÄ…ca,Rozproszone Ĺ›wiatĹ‚o,Umiarkowanie,0
1,Adiantum raddianum,Fern,Bright light,Diffused,Keep moist between watering. Must not be dry b...,30,12,nietoksyczna,nietoksyczna,Tylko temperatura pokojowa,Normalna tolerancja ciepĹ‚a,0,0,PaproÄ‡ / roĹ›lina zielona,Rozproszone Ĺ›wiatĹ‚o,CzÄ™sto,0
2,Aechmea fatsiata,Bromeliad,Bright light,Diffused,Water when soil is half dry. Change water in t...,30,12,nietoksyczna,moĹĽe powodowaÄ‡ lekkie podraĹĽnienia,Tylko temperatura pokojowa,Normalna tolerancja ciepĹ‚a,0,0,RoĹ›lina kwitnÄ…ca,Rozproszone Ĺ›wiatĹ‚o,Umiarkowanie,1


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 159 entries, 0 to 158
Data columns (total 17 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   latin              159 non-null    object
 1   category           159 non-null    object
 2   ideallight         159 non-null    object
 3   toleratedlight     159 non-null    object
 4   watering           159 non-null    object
 5   tempmax.celsius    159 non-null    int64 
 6   tempmin.celsius    159 non-null    int64 
 7   toxicity_animals   159 non-null    object
 8   toxicity_humans    159 non-null    object
 9   tempmin_pasmo      159 non-null    object
 10  tempmax_pasmo      159 non-null    object
 11  is_cold_tolerant   159 non-null    int64 
 12  is_heat_tolerant   159 non-null    int64 
 13  category_group     159 non-null    object
 14  light_level_clean  159 non-null    object
 15  watering_group     159 non-null    object
 16  toxicity_any       159 non-null    int64 
dt

In [4]:
# ## Wybór 8 kolumn na podstawie feature_list.json
import json
from pathlib import Path

# ścieżka do pliku feature_list.json
NOTEBOOK_DIR = Path(__file__).resolve().parent if "__file__" in locals() else Path.cwd()
FEATURE_LIST = NOTEBOOK_DIR.parent / "artifacts" / "feature_list.json"

# wczytanie specyfikacji cech
with open(FEATURE_LIST, "r", encoding="latin") as f:
    feature_spec = json.load(f)

# lista kolumn w odpowiedniej kolejności
features_order = [f["name"] for f in feature_spec["features_order"]]

X = df[features_order].copy()
print("✅ Wczytano:", FEATURE_LIST)
print("Shape X:", X.shape)
X.head(3)


✅ Wczytano: c:\Users\annas\projekty_infoshare\projekt ML\artifacts\feature_list.json
Shape X: (159, 8)


,tempmin_pasmo,tempmax_pasmo,is_cold_tolerant,is_heat_tolerant,category_group,light_level_clean,watering_group,toxicity_any
0,Lubi ciepĹ‚o,Normalna tolerancja ciepĹ‚a,0,0,WiszÄ…ca,Rozproszone Ĺ›wiatĹ‚o,Umiarkowanie,0
1,Tylko temperatura pokojowa,Normalna tolerancja ciepĹ‚a,0,0,PaproÄ‡ / roĹ›lina zielona,Rozproszone Ĺ›wiatĹ‚o,CzÄ™sto,0
2,Tylko temperatura pokojowa,Normalna tolerancja ciepĹ‚a,0,0,RoĹ›lina kwitnÄ…ca,Rozproszone Ĺ›wiatĹ‚o,Umiarkowanie,1



Wybrałam **5-Fold CV** zamiast pojedynczego train/test split, aby zmaksymalizować wykorzystanie danych
i uzyskać bardziej wiarygodną ocenę jakości rekomendacji.


In [5]:
from sklearn.model_selection import KFold

# przygotowanie 5-fold CV
kf = KFold(n_splits=5, shuffle=True, random_state=42)

for fold, (train_idx, test_idx) in enumerate(kf.split(X), 1):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    print(f"Fold {fold}: train={len(X_train)}, test={len(X_test)}")


Fold 1: train=127, test=32
Fold 2: train=127, test=32
Fold 3: train=127, test=32
Fold 4: train=127, test=32
Fold 5: train=128, test=31


In [9]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

categorical_features = [
    f["name"] for f in feature_spec["features_order"]
    if f.get("encoding") == "onehot" or f.get("type") == "categorical"
]

numeric_features = [
    f["name"] for f in feature_spec["features_order"]
    if f.get("type") == "numeric"
]

preprocessor_ohe = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical_features),
        ("num", "passthrough", numeric_features),
    ],
    remainder="drop"
)

# 5-fold CV
for fold, (tr_idx, te_idx) in enumerate(kf.split(X), 1):
    X_tr, X_te = X.iloc[tr_idx], X.iloc[te_idx]
    X_tr_ohe = preprocessor_ohe.fit_transform(X_tr)
    X_te_ohe = preprocessor_ohe.transform(X_te)
    print(f"Fold {fold} → X_tr_ohe: {X_tr_ohe.shape}, X_te_ohe: {X_te_ohe.shape}")


Fold 1 → X_tr_ohe: (127, 21), X_te_ohe: (32, 21)
Fold 2 → X_tr_ohe: (127, 21), X_te_ohe: (32, 21)
Fold 3 → X_tr_ohe: (127, 21), X_te_ohe: (32, 21)
Fold 4 → X_tr_ohe: (127, 21), X_te_ohe: (32, 21)
Fold 5 → X_tr_ohe: (128, 21), X_te_ohe: (31, 21)


## One-Hot Encoding (OHE) z ColumnTransformer

W tym kroku zastosowałam **One-Hot Encoding** tylko do cech kategorycznych
(`tempmin_pasmo`, `tempmax_pasmo`, `category_group`, `light_level_clean`, `watering_group`).

### Dlaczego ColumnTransformer?
- Umożliwia zdefiniowanie, które kolumny mają być poddane OHE.
- Trzyma informację o transformacjach, co ułatwia późniejsze łączenie
  z innymi krokami pipeline (np. skalowaniem cech numerycznych).
- Daje dostęp do nazw cech po zakodowaniu (poprzez `get_feature_names_out`).

### Dlaczego handle_unknown="ignore"?

Używamy handle_unknown="ignore" w OneHotEncoder, aby nie wywalało błędu, gdy w danych testowych/inference pojawi się kategoria, której nie było w treningu. Nie zmienia to liczby kolumn ani nazw cech, tylko daje stabilność.

### Wynik
- Po zakodowaniu otrzymujemy **18 kolumn** (cechy binarne dla wszystkich kategorii).
- Dla każdego folda 5-CV mamy macierze np.:
  - Fold 1 → `X_train`: (127, 18), `X_test`: (32, 18)
  - Fold 5 → `X_train`: (128, 18), `X_test`: (31, 18)

**Podsumowanie:**  
Dzięki użyciu `ColumnTransformer` z OHE dane kategoryczne są teraz w formacie numerycznym,
gotowym do dalszego przetwarzania.


In [ ]:
# ## Nazwy cech po OHE (tylko dla preprocessor_ohe i tylko 1. fold)
import json
from pathlib import Path
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer

ARTIFACTS_DIR = Path("artifacts")
ARTIFACTS_DIR.mkdir(exist_ok=True, parents=True)

# Definicja preprocesora z OHE odpornym na nieznane kategorie
ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=True)
preprocessor_ohe = ColumnTransformer(
    transformers=[("cat", ohe, categorical_features)],
    remainder="drop"
)

for fold, (tr_idx, te_idx) in enumerate(kf.split(X), 1):
    X_tr, X_te = X.iloc[tr_idx], X.iloc[te_idx]
    X_tr_ohe = preprocessor_ohe.fit_transform(X_tr)  # fit tylko na train
    X_te_ohe = preprocessor_ohe.transform(X_te)

    if fold == 1:
        ohe_names = preprocessor_ohe.named_transformers_["cat"] \
            .get_feature_names_out(categorical_features).tolist()
        print("Liczba nazw:", len(ohe_names))
        print("Przykład:", ohe_names[:10])
        with open(ARTIFACTS_DIR / "feature_names_ohe.json", "w", encoding="utf-8") as f:
            json.dump(ohe_names, f, ensure_ascii=False, indent=2)
    break  # tylko pierwszy fold dla nazw


Liczba nazw: 18
Przykład: ['tempmin_pasmo_Lubi ciepło', 'tempmin_pasmo_Odporna na chłód', 'tempmin_pasmo_Toleruje chłód', 'tempmin_pasmo_Tylko temperatura pokojowa', 'tempmax_pasmo_Normalna tolerancja ciepła', 'tempmax_pasmo_Odporna na upał', 'tempmax_pasmo_Wrażliwa na upał', 'category_group_Inne', 'category_group_Palma / drzewko ozdobne', 'category_group_Paproć / roślina zielona']


In [ ]:
# %% Fit & dump OneHotEncoder na całym zbiorze + zapis nazw
import json
from pathlib import Path
import joblib
import pandas as pd
import sklearn
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer

# --- Ścieżki / katalogi
ARTIFACTS_DIR = Path("artifacts")
ARTIFACTS_DIR.mkdir(exist_ok=True, parents=True)

CSV = "plants_clean_final.csv"
OHE_PKL = ARTIFACTS_DIR / "ohe.pkl"
OHE_NAMES_JSON = ARTIFACTS_DIR / "feature_names_ohe.json"

# --- Dane i lista kolumn kategorycznych
df = pd.read_csv(CSV)
CAT_COLS = ["tempmin_pasmo","tempmax_pasmo","category_group","light_level_clean","watering_group"]

# --- OHE z handle_unknown="ignore" (obsługa różnic wersji sklearn)
sk_version = tuple(int(x) for x in sklearn.__version__.split(".")[:2])  # (major, minor)
if sk_version >= (1, 2):
    ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=True)
else:
    ohe = OneHotEncoder(handle_unknown="ignore", sparse=True)

preprocessor_ohe = ColumnTransformer(
    transformers=[("cat", ohe, CAT_COLS)],
    remainder="drop",
)

# --- Fit na CAŁYM zbiorze (zamrożenie encoder-a)
preprocessor_ohe.fit(df[CAT_COLS])

# --- Zapis obiektu do pliku (do inference w apce)
joblib.dump(preprocessor_ohe, OHE_PKL)

# --- Eksport nazw kolumn po OHE
ohe_names = preprocessor_ohe.named_transformers_["cat"].get_feature_names_out(CAT_COLS).tolist()
with open(OHE_NAMES_JSON, "w", encoding="utf-8") as f:
    json.dump(ohe_names, f, ensure_ascii=False, indent=2)

# --- Szybki smoke test
assert len(ohe_names) == 18, f"Oczekiwano 18 kolumn OHE, jest {len(ohe_names)}"
X_ohe = preprocessor_ohe.transform(df[CAT_COLS])
assert X_ohe.shape[1] == 18, f"Oczekiwano 18 kolumn po transformacji, jest {X_ohe.shape[1]}"

print("✔ OHE zamrożony i zapisany do", OHE_PKL)
print("✔ Nazwy kolumn zapisane do", OHE_NAMES_JSON)


✔ OHE zamrożony i zapisany do artifacts\ohe.pkl
✔ Nazwy kolumn zapisane do artifacts\feature_names_ohe.json


#### Notatka — zamrożenie OHE (`ohe.pkl`)
- Cel: mieć **stabilny encoder** do inference (apka) z tym samym układem 18 kolumn.
- Ustawienie: `OneHotEncoder(handle_unknown="ignore")` — ignoruje nowe kategorie (stabilność, brak błędów).
- Proces: fit na **całym zbiorze**, zapis `artifacts/ohe.pkl` + eksport `artifacts/feature_names_ohe.json`.
- W inference: ładujemy `ohe.pkl` → `.transform()` na danych kategorycznych → mamy stałą macierz 18 kolumn.


In [ ]:
# %% Smoke test: OHE artifacts + nazwy
import json
from pathlib import Path
import joblib
import pandas as pd

ART = Path("artifacts")
CSV = "plants_clean_final.csv"
OHE_PKL = ART / "ohe.pkl"
OHE_NAMES_JSON = ART / "feature_names_ohe.json"

# 1) Wczytaj dane i artefakty
df = pd.read_csv(CSV)
with open(OHE_NAMES_JSON, "r", encoding="utf-8") as f:
    names_json = json.load(f)

ohe = joblib.load(OHE_PKL)

# 2) Zdefiniuj kolumny kategoryczne (kolejność MUSI odpowiadać treningowi)
CAT_COLS = ["tempmin_pasmo","tempmax_pasmo","category_group","light_level_clean","watering_group"]

# 3) Brak NaN w kategoriach
assert df[CAT_COLS].isna().sum().sum() == 0, "NaNy w kolumnach kategorycznych!"

# 4) Transformacja działa i ma stały wymiar
X_ohe = ohe.transform(df[CAT_COLS])
assert X_ohe.shape[1] == len(names_json), "Liczba kolumn OHE ≠ lista w JSON!"

# 5) Zgodność nazw z modelem (źródło prawdy = fitted encoder)
names_model = ohe.named_transformers_["cat"].get_feature_names_out(CAT_COLS).tolist()
assert names_model == names_json, "feature_names_ohe.json ≠ get_feature_names_out()!"

# 6) Szybki sanity-check: każdy wiersz ma dokładnie sumę 5 jedynek (1 kolumna aktywna na każdą z 5 cech)
# (Jeśli w którejś kolumnie masz brak kategorii, suma może wyjść <5; wtedy to sygnał do sprawdzenia danych)
row_sums = (X_ohe.toarray() > 0).sum(axis=1)
assert (row_sums >= 5).all(), "Niektóre wiersze mają mniej niż 5 aktywnych kategorii po OHE!"

print("✅ OHE smoke test: artefakty i nazwy OK, transformacja OK, wymiary OK")


✅ OHE smoke test: artefakty i nazwy OK, transformacja OK, wymiary OK


Ten test upewnia, że ohe.pkl i feature_names_ohe.json są ze sobą w 100% zgodne i że transformacja zwraca stałą macierz 18 kolumn — dzięki temu inference w apce jest stabilne.

In [ ]:
# %% Inference: stała kolejność cech (OHE → num/flag)
import json, joblib, numpy as np, pandas as pd
from pathlib import Path
from scipy import sparse

ART = Path("artifacts")
CSV = "plants_clean_final.csv"
FEATURE_LIST_JSON = "feature_list.json"
OHE_PKL = ART / "ohe.pkl"
OHE_NAMES_JSON = ART / "feature_names_ohe.json"

# 1) Wczytaj definicję kolejności z feature_list.json
with open(FEATURE_LIST_JSON, "r", encoding="latin-1") as f:
    flist = json.load(f)

CAT_COLS = [it["name"] for it in flist["features_order"]
            if it.get("type") == "categorical" and it.get("encoding") == "onehot"]

NUM_FLAG_COLS = [it["name"] for it in flist["features_order"]
                 if not (it.get("type") == "categorical" and it.get("encoding") == "onehot")]

# 2) Artefakty OHE
ohe = joblib.load(OHE_PKL)
with open(OHE_NAMES_JSON, "r", encoding="utf-8") as f:
    ohe_names = json.load(f)  # 18 nazw OHE w stałej kolejności

# 3) Budowa macierzy cech w stałej kolejności
def build_feature_matrix(df: pd.DataFrame):
    # a) kategorie → OHE (18 kolumn, kolejność = ohe_names)
    X_cat = ohe.transform(df[CAT_COLS])  # sparse CSR

    # b) num/flag → numpy → sparse (kolejność = NUM_FLAG_COLS)
    X_num = df[NUM_FLAG_COLS].to_numpy(dtype=float) if NUM_FLAG_COLS else np.empty((len(df), 0))
    X_num = sparse.csr_matrix(X_num)

    # c) sklej: najpierw OHE, potem num/flag
    X = sparse.hstack([X_cat, X_num], format="csr")

    # d) komplet nazw
    feature_names_full = ohe_names + NUM_FLAG_COLS

    # e) sanity checks
    assert X.shape[1] == len(feature_names_full), "Mismatch: X kolumn ≠ liczba nazw!"
    return X, feature_names_full

# 4) Przykład użycia
df = pd.read_csv(CSV)
X, feature_names_full = build_feature_matrix(df)
print("Shape:", X.shape)
print("Pierwsze 5:", feature_names_full[:5])
print("Ostatnie 5:", feature_names_full[-5:])


Shape: (159, 21)
Pierwsze 5: ['tempmin_pasmo_Lubi ciepło', 'tempmin_pasmo_Odporna na chłód', 'tempmin_pasmo_Toleruje chłód', 'tempmin_pasmo_Tylko temperatura pokojowa', 'tempmax_pasmo_Normalna tolerancja ciepła']
Ostatnie 5: ['watering_group_Rzadko', 'watering_group_Umiarkowanie', 'is_cold_tolerant', 'is_heat_tolerant', 'toxicity_any']


**Notatka — kolejność cech na inference**
- Kolejność definiujemy w `feature_list.json`.
- Budujemy wektor: [OHE (18 kolumn) → num/flag (wg listy)].
- Dzięki temu każdy wektor wejściowy ma identyczny układ, co gwarantuje stabilne działanie modelu/apki.


✅ Wniosek: konfiguracja inference jest spójna, wektor wejściowy ma stały układ (18 OHE + 3 num/flag = 21 cech).

In [ ]:
# %% Fit & dump StandardScaler (tylko dla cold/heat tolerant)
import joblib
import pandas as pd
from sklearn.preprocessing import StandardScaler
from pathlib import Path

ART = Path("artifacts")
CSV = "plants_clean_final.csv"
SCALER_PKL = ART / "scaler.pkl"

# --- Dane
df = pd.read_csv(CSV)

# --- Skalowane kolumny
SCALE_COLS = ["is_cold_tolerant", "is_heat_tolerant"]

# --- Kolumna binarna zostaje nietknięta
BIN_COLS = ["toxicity_any"]

# --- Scaler
scaler = StandardScaler()
scaler.fit(df[SCALE_COLS])   # fit tylko na 2 kolumnach

# --- Zapis
joblib.dump(scaler, SCALER_PKL)

# --- Smoke test
X_scaled = scaler.transform(df[SCALE_COLS])
print("Shape:", X_scaled.shape)
print("Średnia ~0:", X_scaled.mean(axis=0))
print("Odchylenie ~1:", X_scaled.std(axis=0))
print("✔ Scaler zapisany do", SCALER_PKL)


Shape: (159, 2)
Średnia ~0: [ 1.11720556e-17 -5.58602780e-18]
Odchylenie ~1: [1. 1.]
✔ Scaler zapisany do artifacts\scaler.pkl


In [ ]:
# %% Inference: budowa wektora 21 cech (OHE + scaler + binarka)
import joblib, json, pandas as pd, numpy as np
from pathlib import Path
from scipy import sparse

ART = Path("artifacts")
CSV = "plants_clean_final.csv"

OHE_PKL = ART / "ohe.pkl"
SCALER_PKL = ART / "scaler.pkl"
OHE_NAMES_JSON = ART / "feature_names_ohe.json"
FEATURE_LIST_JSON = "feature_list.json"

# --- Wczytaj artefakty
ohe = joblib.load(OHE_PKL)
scaler = joblib.load(SCALER_PKL)
with open(OHE_NAMES_JSON, "r", encoding="utf-8") as f:
    ohe_names = json.load(f)

# --- Definicja kolumn
CAT_COLS = ["tempmin_pasmo","tempmax_pasmo","category_group","light_level_clean","watering_group"]
SCALE_COLS = ["is_cold_tolerant","is_heat_tolerant"]
BIN_COLS = ["toxicity_any"]

# --- Funkcja do budowy pełnego wektora cech
def build_feature_matrix(df: pd.DataFrame):
    # 1) Kategoryczne → OHE (18 kolumn)
    X_cat = ohe.transform(df[CAT_COLS])

    # 2) Numeryczne/flagowe → scaler (2 kolumny)
    X_num = scaler.transform(df[SCALE_COLS])
    X_num = sparse.csr_matrix(X_num)

    # 3) Binarka → bez zmian (1 kolumna)
    X_bin = sparse.csr_matrix(df[BIN_COLS].to_numpy(dtype=float))

    # 4) Sklej: OHE + num + bin
    X = sparse.hstack([X_cat, X_num, X_bin], format="csr")

    # 5) Pełna lista nazw cech
    feature_names_full = ohe_names + SCALE_COLS + BIN_COLS
    assert X.shape[1] == len(feature_names_full), "Rozjazd kolumn!"

    return X, feature_names_full

# --- Przykład użycia
df = pd.read_csv(CSV)
X_full, feature_names_full = build_feature_matrix(df)

print("Shape:", X_full.shape)       # (159, 21)
print("Pierwsze 5:", feature_names_full[:5])
print("Ostatnie 5:", feature_names_full[-5:])


Shape: (159, 21)
Pierwsze 5: ['tempmin_pasmo_Lubi ciepło', 'tempmin_pasmo_Odporna na chłód', 'tempmin_pasmo_Toleruje chłód', 'tempmin_pasmo_Tylko temperatura pokojowa', 'tempmax_pasmo_Normalna tolerancja ciepła']
Ostatnie 5: ['watering_group_Rzadko', 'watering_group_Umiarkowanie', 'is_cold_tolerant', 'is_heat_tolerant', 'toxicity_any']


**Notatka — budowa wektora cech (21)**
- Inference = najpierw OHE (18 kolumn), potem skalowane (2), na końcu binarka (1).
- Efekt: zawsze stabilny wektor 21 cech.
- To jest „pełny obrazek” rośliny, gotowy do dalszej redukcji (np. PCA).
- Decyzja projektowa: nie eksportujemy X_full


In [ ]:
# %% Fit & dump PCA (embedding roślin)
import joblib
import pandas as pd
from sklearn.decomposition import PCA
from pathlib import Path

ART = Path("artifacts")
CSV = "plants_clean_final.csv"
PCA_PKL = ART / "pca.pkl"

# --- Wczytaj dane i zbuduj macierz cech
df = pd.read_csv(CSV)
X_full, feature_names_full = build_feature_matrix(df)   # używamy funkcji z poprzedniego kroku

# --- PCA (np. 10 wymiarów, można dostosować)
pca = PCA(n_components=10, random_state=42)
X_pca = pca.fit_transform(X_full.toarray())

# --- Zapis
joblib.dump(pca, PCA_PKL)

# --- Podgląd wyników
print("Shape oryginalny:", X_full.shape)
print("Shape po PCA:", X_pca.shape)
print("Wyjaśniona wariancja [%]:", (pca.explained_variance_ratio_*100).round(2))


Shape oryginalny: (159, 21)
Shape po PCA: (159, 10)
Wyjaśniona wariancja [%]: [32.15 20.7  10.32  7.79  7.32  5.91  4.57  2.86  2.5   1.9 ]


## interpretacja wyników PCA

Shape oryginalny (159, 21) → dokładnie 21 cech wejściowych.

Shape po PCA (159, 10) → embedding 10-wymiarowy, czyli każda roślina ma wektor 10 liczb zamiast 21.

Wyjaśniona wariancja [%] → pokazuje, ile informacji (różnorodności danych) „łapie” każda składowa:

PC1: 32.15%

PC2: 20.7%

PC3: 10.32%

PC4–PC10: po kilka procent
→ razem 10 składowych daje ~96% całej informacji 

In [ ]:
# %% Inference: OHE + Scaler + Binarka → 21 cech → PCA(10D)
import json, joblib, numpy as np, pandas as pd
from pathlib import Path
from scipy import sparse

ART = Path("artifacts")
OHE_PKL = ART / "ohe.pkl"
SCALER_PKL = ART / "scaler.pkl"       # scaler tylko dla 2 kolumn (cold/heat)
PCA_PKL = ART / "pca.pkl"             # wcześniej dopasowany PCA (n_components=10)
OHE_NAMES_JSON = ART / "feature_names_ohe.json"

# --- Załaduj artefakty
ohe = joblib.load(OHE_PKL)
scaler = joblib.load(SCALER_PKL)
pca = joblib.load(PCA_PKL)
with open(OHE_NAMES_JSON, "r", encoding="utf-8") as f:
    ohe_names = json.load(f)

# --- Definicje kolumn (ustalone wcześniej)
CAT_COLS   = ["tempmin_pasmo","tempmax_pasmo","category_group","light_level_clean","watering_group"]
SCALE_COLS = ["is_cold_tolerant","is_heat_tolerant"]   # skalujemy
BIN_COLS   = ["toxicity_any"]                          # NIE skalujemy

def build_feature_matrix(df: pd.DataFrame):
    """Zwraca X_full (CSR) 21 kolumn + listę nazw w stałej kolejności."""
    # 1) kategoryczne → OHE (18 kolumn)
    X_cat = ohe.transform(df[CAT_COLS])
    # 2) num → scaler (2 kolumny)
    X_num = scaler.transform(df[SCALE_COLS])
    X_num = sparse.csr_matrix(X_num)
    # 3) binarka (1 kolumna)
    X_bin = sparse.csr_matrix(df[BIN_COLS].to_numpy(dtype=float))
    # 4) sklej
    X_full = sparse.hstack([X_cat, X_num, X_bin], format="csr")
    feature_names_full = ohe_names + SCALE_COLS + BIN_COLS
    assert X_full.shape[1] == len(feature_names_full), "Rozjazd liczby kolumn!"
    return X_full, feature_names_full

def to_embedding(df: pd.DataFrame):
    """Buduje 21-cech i przekształca przez PCA → zwraca macierz (n × 10)."""
    X_full, feature_names_full = build_feature_matrix(df)
    X_pca = pca.transform(X_full.toarray())  # embedding 10D
    return X_pca, feature_names_full

# --- Przykład: wiele rekordów (z CSV)
df = pd.read_csv("plants_clean_final.csv")
X_pca, names = to_embedding(df)
print("Embedding shape (batch):", X_pca.shape)  # (n, 10)

# --- Przykład: pojedynczy rekord (dict → DataFrame)
sample = {
    "tempmin_pasmo": "Lubi ciepło",
    "tempmax_pasmo": "Normalna tolerancja ciepła",
    "category_group": "Roślina kwitnąca",
    "light_level_clean": "Dużo światła",
    "watering_group": "Rzadko",
    "is_cold_tolerant": 0,
    "is_heat_tolerant": 1,
    "toxicity_any": 1,
}
X_pca_one, _ = to_embedding(pd.DataFrame([sample]))
print("Embedding shape (one):", X_pca_one.shape)  # (1, 10)
print("Embedding (1. roślina):", np.round(X_pca_one[0], 3))


Embedding shape (batch): (159, 10)
Embedding shape (one): (1, 10)
Embedding (1. roślina): [ 1.756 -1.345  0.598 -0.026  1.39  -0.683  0.3   -0.073  0.114 -0.388]


Embedding shape (batch): (159, 10) → każda z 159 roślin ma teraz swój wektor w przestrzeni 10-wymiarowej. To dokładnie to, co chcieliśmy osiągnąć PCA.

Embedding shape (one): (1, 10) → dla jednej rośliny dostajesz jeden wektor (10 liczb).

Embedding (np. pierwsza roślina): [ 1.756 -1.345 0.598 -0.026 1.39 -0.683 0.3 -0.073 0.114 -0.388]

to są współrzędne tej rośliny w „mapie roślin” stworzonej przez PCA,

każda liczba to udział danej składowej głównej (PC1, PC2, …, PC10),

takie embeddingi będą używane do:

klastrowania (KMeans pogrupuje rośliny w podobne klastry),

wyszukiwania podobieństwa (np. cosine similarity → top-N podobnych roślin).

In [43]:
# %% Cosine similarity na embeddingach PCA
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

# --- Wczytaj dane i embeddingi
df = pd.read_csv("plants_clean_final.csv")
X_pca, _ = to_embedding(df)   # używamy funkcji to_embedding z poprzedniego kroku (21 cech -> PCA 10D)

# --- Oblicz macierz cosine similarity (159 x 159)
sim_matrix = cosine_similarity(X_pca)

# --- Funkcja: top-N podobnych roślin do wybranej
def top_n_similar(latin_name, n=5):
    # znajdź indeks rośliny
    idx = df.index[df["latin"] == latin_name].tolist()
    if not idx:
        raise ValueError(f"Roślina {latin_name} nieznaleziona w danych")
    idx = idx[0]
    
    # weź podobieństwa do wszystkich
    sims = sim_matrix[idx]
    
    # posortuj wg podobieństwa (od największego do najmniejszego), pomijając siebie (idx)
    similar_idx = np.argsort(-sims)
    similar_idx = [i for i in similar_idx if i != idx][:n]
    
    # zwróć tabelkę
    result = pd.DataFrame({
        "latin": df.loc[similar_idx, "latin"].values,
        "similarity": sims[similar_idx].round(3),
        "category_group": df.loc[similar_idx, "category_group"].values
    })
    return result

# --- Przykład użycia
plant = df["latin"].iloc[0]   # weź pierwszą roślinę z tabeli
print("Roślina bazowa:", plant)
print(top_n_similar(plant, n=5))


Roślina bazowa: Aeschynanthus lobianus
                 latin  similarity           category_group
0      Licuala spinosa       0.829  Palma / drzewko ozdobne
1  Tillandsia Creation       0.802         Roślina kwitnąca
2    Tillandsia cyanea       0.802         Roślina kwitnąca
3   Vriesea Christiane       0.802         Roślina kwitnąca
4    Vriesea splendens       0.802         Roślina kwitnąca


**Cosine similarity**
- Cosine similarity = miara podobieństwa między embeddingami (1 = identyczne, 0 = brak podobieństwa).
- W projekcie: używamy embeddingów PCA (10D).
- Wynik: dla wybranej rośliny możemy zwrócić top-N najbardziej podobnych gatunków.
- To podstawowy mechanizm rekomendacji w Plantelligence.

✅ Cosine similarity przeszedł test pomyślnie

In [44]:
# %% Eksport embeddingów PCA do pliku .npy
import numpy as np
import pandas as pd

# --- Wczytaj dane
df = pd.read_csv("plants_clean_final.csv")

# --- Oblicz embeddingi (funkcja to_embedding już zdefiniowana)
X_pca, _ = to_embedding(df)   # (159, 10)

# --- Zapis do pliku .npy
np.save("artifacts/plant_embeddings.npy", X_pca)

print("✔ Zapisano embeddingi:", X_pca.shape, "-> artifacts/plant_embeddings.npy")



✔ Zapisano embeddingi: (159, 10) -> artifacts/plant_embeddings.npy


In [45]:
# %% Eksport embeddingów + nazwy roślin do CSV (łatwiejszy podgląd)
emb_df = pd.DataFrame(X_pca, columns=[f"PC{i+1}" for i in range(X_pca.shape[1])])
emb_df["latin"] = df["latin"]
emb_df["category_group"] = df["category_group"]

emb_df.to_csv("artifacts/plant_embeddings.csv", index=False)

print("✔ Zapisano embeddingi z metadanymi -> artifacts/plant_embeddings.csv")


✔ Zapisano embeddingi z metadanymi -> artifacts/plant_embeddings.csv


**Eksport embeddingów**
- Zapisano macierz PCA (159 × 10) do pliku `plant_embeddings.npy`.
- Ten plik jest źródłem embeddingów roślin dla dalszych kroków (KMeans, similarity search).
- Dodatkowo: zapisano wersję z nazwami (`plant_embeddings.csv`) do podglądu i debugowania.
